# 07. Module system — `BaseModule`, the canonical stack, and custom modules

ArcLLM's module system is a stack of middleware around an `LLMProvider`
adapter. Each module wraps an `_inner` provider, exposes the same
`invoke(messages, tools, **kwargs)` contract, and adds one concern —
retry, fallback, queueing, telemetry, audit, redaction, OTel spans,
circuit breaking, classification routing, or rate limiting.

`load_model()` (covered in `arcllm/04-agentic-loop.ipynb`) is the
canonical way to compose them. **This notebook is the deep dive on
what those modules *are*** — the `BaseModule` contract, why the
default stack order is what it is, and how to write your own module
and slot it into the stack.

You will learn:

- The `BaseModule` contract — `_inner`, `__call__`-style `invoke`,
  `validate_config_keys`, the OTel span helper, what subclasses must
  override
- The complete catalog of all 11 official modules and where each
  one sits in the stack
- The canonical wrap order — `Otel → Queue → Telemetry → Audit →
  Security → CircuitBreaker → Retry → Fallback → RateLimit →
  [Routing|Adapter]` — and *why* every layer sits where it does
- How to write a custom module from scratch (a request labeler) and
  slot it into the stack manually
- A demo of why ordering matters — moving `RateLimit` outside
  `Retry` changes the observable failure pattern
- A `describe_stack()` helper for inspecting the live `_inner`
  chain on a constructed model

If you have not already read `arcllm/04-agentic-loop.ipynb`, start
there — it walks every kwarg of `load_model()`. This notebook builds
on that and goes underneath.

## 1. Setup

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Pull in the public surface — every officially exported module class,
the `BaseModule` contract pieces, the data types, and `load_model()`
/ `clear_cache()` for the integration demos. We also force
`ARC_CONFIG_DIR` to an empty temp directory so this notebook reflects
the *packaged* defaults, not whatever sits in the developer's
`~/.arc/arcllm.toml`.

In [ ]:
import asyncio
import tempfile
from typing import Any
from unittest.mock import AsyncMock, MagicMock, patch

# Isolate from any user-installed arcllm.toml — match the test conftest pattern.
os.environ["ARC_CONFIG_DIR"] = tempfile.mkdtemp(prefix="arc-cfg-07-")
os.environ.setdefault("ANTHROPIC_API_KEY", "test-key-not-real")
os.environ.setdefault("OPENAI_API_KEY", "test-key-not-real")

import arcllm
from arcllm import (
    AuditModule,
    BaseModule,
    FallbackModule,
    LLMResponse,
    Message,
    OtelModule,
    QueueModule,
    RateLimitModule,
    RetryModule,
    SecurityModule,
    TelemetryModule,
    Tool,
    Usage,
    clear_cache,
    load_model,
)
from arcllm.exceptions import ArcLLMAPIError, ArcLLMConfigError
from arcllm.modules.base import validate_config_keys
from arcllm.modules.circuit_breaker import CircuitBreakerModule
from arcllm.modules.routing import RoutingModule
from arcllm.types import LLMProvider

print("arcllm version:", arcllm.__version__)

All cells in this notebook are mock-first. We never hit the
network. The pattern we'll use throughout is a `MagicMock` typed
as `LLMProvider`, with the inner `invoke()` scripted as an
`AsyncMock`.

In [ ]:
OK = LLMResponse(
    content="hello from mock",
    usage=Usage(input_tokens=10, output_tokens=5, total_tokens=15),
    model="mock-model",
    stop_reason="end_turn",
)


def make_mock_inner(
    *,
    name: str = "anthropic",
    model: str = "mock-model",
    side_effect: Any = None,
    return_value: LLMResponse | None = None,
) -> LLMProvider:
    """A minimal LLMProvider stand-in for stack composition demos."""
    inner = MagicMock(spec=LLMProvider)
    inner.name = name
    inner.model_name = model
    inner.validate_config.return_value = True
    if side_effect is not None:
        inner.invoke = AsyncMock(side_effect=side_effect)
    else:
        inner.invoke = AsyncMock(return_value=return_value or OK)
    inner.close = AsyncMock()
    return inner


messages = [Message(role="user", content="hi")]
print("helpers ready")

## 2. Module philosophy — middleware around an adapter

Adapters speak HTTP to a provider. They convert `Message` /
`Tool` objects into provider-specific JSON, post the request, and
parse the response into an `LLMResponse`. That is *all* an adapter
does.

Everything else — retrying transient errors, capping concurrency,
redacting PII, recording trace events, breaking circuits on a
wedged provider — is a separate concern. ArcLLM puts each concern
in a **module**: a class that wraps an `LLMProvider` and exposes
the same interface so callers cannot tell the difference.

Three properties matter:

1. **Composable.** Every module *is* an `LLMProvider`. Wrapping
   one module around another is a no-op for the caller.
2. **Independent.** Modules don't reach across each other. Each
   one talks only to `self._inner`.
3. **Optional.** Modules are opt-in via config. A bare adapter is
   a perfectly valid `LLMProvider`.

The result is a stack of middleware around an adapter — the same
decorator pattern Django/Express middleware uses, applied to LLM
calls instead of HTTP requests.

```
agent.invoke(messages, tools)
      │
      ▼
┌──────────────────┐
│ OtelModule       │ ← outermost: wall-clock span around everything
├──────────────────┤
│ QueueModule      │
├──────────────────┤
│ TelemetryModule  │
├──────────────────┤
│ AuditModule      │
├──────────────────┤
│ SecurityModule   │
├──────────────────┤
│ CircuitBreaker   │
├──────────────────┤
│ RetryModule      │
├──────────────────┤
│ FallbackModule   │
├──────────────────┤
│ RateLimitModule  │ ← innermost: last gate before the adapter
├──────────────────┤
│ Adapter (HTTP)   │
└──────────────────┘
```

Each box is one class. Each class is < 200 LOC. Each class can
be tested in isolation by handing it a `MagicMock(spec=LLMProvider)`
as its `_inner`.

The contract every layer obeys is the `LLMProvider` Protocol
(`arcllm/types.py`). It has four members:

In [ ]:
for member in ("name", "model_name", "invoke", "validate_config"):
    print(f"  LLMProvider.{member}")

print()
print("hasattr(BaseModule, 'invoke'):", hasattr(BaseModule, "invoke"))
print("BaseModule subclasses LLMProvider:", issubclass(BaseModule, LLMProvider))

## 3. The `BaseModule` contract

Every official module subclasses `BaseModule`. The base class lives
in `arcllm/modules/base.py` and is intentionally tiny — it gives
subclasses three things:

1. **A constructor signature.** `__init__(self, config, inner)`.
   Every module takes a config `dict` and an inner `LLMProvider`.
2. **Pass-through delegation.** `name`, `model_name`,
   `validate_config()`, `close()`, and `invoke()` all forward to
   `self._inner` by default. Subclasses override only what they
   need.
3. **An OTel span helper.** `self._span(name, attrs)` is a
   context manager that creates a span on the `arcllm` tracer,
   records exceptions, and re-raises. No-op when no SDK is
   configured.

And one helper function — `validate_config_keys(config, valid,
module_name)` — for rejecting unrecognized config keys with a
clear error message. Most modules call it in `__init__`.

In [ ]:
import inspect

src = inspect.getsource(BaseModule)
print(src)

Three things to internalize from that source:

- **`self._inner`** is the convention every subclass uses to reach
  the next layer. Walking `_inner` is how you inspect a live stack
  — we'll do this in section 8.
- **`invoke()`** is the single hook subclasses customise. There
  is no `before_call`/`after_call` pre/post lifecycle — modules
  just override `invoke()` and put their logic before, around, or
  after the `await self._inner.invoke(...)` call.
- **`close()`** propagates by default. If your subclass holds
  resources (a thread pool, a file handle), override it and
  remember to `await self._inner.close()` at the end.

The simplest possible "module" is a literal `BaseModule` instance —
it forwards every call. Useful as a sanity check that the wiring
works without changing behaviour.

In [ ]:
transparent = BaseModule({}, make_mock_inner())

result = await transparent.invoke(messages)
print("name:", transparent.name)
print("model_name:", transparent.model_name)
print("validate_config():", transparent.validate_config())
print("invoke result:", result.content)

`validate_config_keys()` is the second piece every subclass uses.
It's a one-line guard that turns a typo'd config key into a
readable error instead of a silently-ignored value.

In [ ]:
_VALID_KEYS = {"enabled", "label", "max_calls"}

# Happy path: known keys pass.
validate_config_keys({"enabled": True, "max_calls": 5}, _VALID_KEYS, "DemoModule")

# Unknown key: clear error.
try:
    validate_config_keys(
        {"max_calls": 5, "max_callz": 7},  # typo
        _VALID_KEYS,
        "DemoModule",
    )
except ArcLLMConfigError as e:
    print("caught:", e)

What subclasses **must** do, in order:

1. Call `super().__init__(config, inner)` so `_config` and
   `_inner` get stored.
2. Call `validate_config_keys(config, _VALID_KEYS, ModuleName)` to
   reject typos.
3. Pull values out of `config` with `.get(key, default)` and store
   them as instance attributes — never re-read `_config` per
   call.
4. Override `invoke()` to add behaviour around `await
   self._inner.invoke(...)`.

What subclasses *may* do:

- Override `close()` if they hold resources.
- Use `self._span(...)` for OTel spans.
- Inject metadata into `kwargs` before forwarding to `_inner`
  (e.g. `RetryModule` injects `_retry_attempt` so `TelemetryModule`
  can record it).

## 4. The module catalog — all 11 modules

Eleven modules ship with arcllm v0.4.0. Every one is a
`BaseModule` subclass; every one is opt-in via a `load_model()`
kwarg or `arcllm.toml` block.

| #  | Module                  | Concern                                                       | Public name in `arcllm.modules` | Deep-dive notebook |
|----|--------------------------|----------------------------------------------------------------|----------------------------------|--------------------|
| 1  | `OtelModule`             | OpenTelemetry distributed traces around the whole call         | `arcllm.OtelModule`              | `11-otel-export`    |
| 2  | `QueueModule`            | Bounded concurrency + backpressure (`asyncio.Semaphore`)       | `arcllm.QueueModule`             | `15-queue-circuit-breaker` |
| 3  | `TelemetryModule`        | Tokens / cost / phase timings; `on_event` + `trace_store`      | `arcllm.TelemetryModule`         | `09-telemetry-module` |
| 4  | `AuditModule`            | Structured audit logs (PII-safe metadata) for compliance       | `arcllm.AuditModule`             | `10-audit-trail`     |
| 5  | `SecurityModule`         | PII redaction + HMAC request signing                           | `arcllm.SecurityModule`          | `12-security-layer`  |
| 6  | `CircuitBreakerModule`   | Per-provider state machine: CLOSED / OPEN / HALF_OPEN          | `arcllm.modules.circuit_breaker` | `15-queue-circuit-breaker` |
| 7  | `RetryModule`            | Exponential backoff + jitter on transient errors               | `arcllm.RetryModule`             | (this notebook)      |
| 8  | `FallbackModule`         | Provider-chain failover with on-demand adapter creation        | `arcllm.FallbackModule`          | (this notebook)      |
| 9  | `RateLimitModule`        | Token-bucket limiter, per-provider, innermost                  | `arcllm.RateLimitModule`         | `08-rate-limiter`    |
| 10 | `RoutingModule`          | Classification-based dispatch (replaces adapter at innermost)  | `arcllm.modules.routing`         | `17-routing-module`  |
| 11 | `BaseModule`             | The transparent base every module subclasses                   | `arcllm.BaseModule`              | (this notebook)      |

`BaseModule` is included in the count because it's a real,
instantiable class — not just an abstract scaffold. You can wrap
an adapter in `BaseModule` to get an inert pass-through (useful
for tests).

Confirm the export list directly:

In [ ]:
import arcllm.modules as mods

print("arcllm.modules.__all__:")
for name in mods.__all__:
    cls = getattr(mods, name)
    print(f"  {name:<22} subclass-of-BaseModule={issubclass(cls, BaseModule)}")

# Two more modules live as siblings (used via load_model / direct import):
from arcllm.modules import circuit_breaker, routing

print(
    f"  CircuitBreakerModule   subclass-of-BaseModule={issubclass(circuit_breaker.CircuitBreakerModule, BaseModule)}"
)
print(
    f"  RoutingModule          subclass-of-BaseModule={issubclass(routing.RoutingModule, BaseModule)}"
)

Every dedicated notebook listed above goes deeper on the module
in question — its config keys, its OTel span attributes, its
edge cases, its tests. This notebook stays at the architecture
layer.

## 5. The canonical stack order — and why

When you call `load_model(...)` with the default kwargs, the
registry composes modules in a fixed order. Reading from the
**outside in** (the order kwargs from a caller arrive at):

```
Otel → Queue → Telemetry → Audit → Security → CircuitBreaker →
       Retry → Fallback → RateLimit → [Routing | Adapter]
```

Equivalently, reading **innermost first** (the order
`registry._build_*` actually wraps in `load_model()`):

```
RateLimit → Fallback → Retry → CircuitBreaker → Security →
          Audit → Telemetry → Queue → Otel
```

(Routing replaces the adapter at the innermost position; it does
not stack alongside the others.)

Why this order? Each layer's position reflects what it needs to
*observe* and what it needs to *protect*:

| Layer (outer→inner) | Job | Why this position |
|----------------------|-----|-------------------|
| **Otel** | Wall-clock span | Outermost so the span includes everything — queue wait, retries, fallback hops. The user sees one span per logical call. |
| **Queue** | Backpressure / concurrency cap | Just inside Otel so OTel observes wait time, but outside telemetry so telemetry isn't billed for queue wait. |
| **Telemetry** | Tokens, cost, phase timings | Below Queue (so wait isn't counted as latency) but above Audit/Security/Retry so it sees the *effective* call result, including retries-as-one-event. |
| **Audit** | Structured compliance logs | Above Security so audit logs the post-redaction view. Audit must observe the same payload security saw. |
| **Security** | PII redaction, request signing | Above CircuitBreaker so a tripped breaker doesn't bypass redaction (defence in depth) and below Audit for ordering above. |
| **CircuitBreaker** | Fail fast on wedged providers | Above Retry so a tripped breaker rejects without retries. Below Security so signed requests are evaluated by the breaker. |
| **Retry** | Exponential backoff on transient errors | Above Fallback so one logical call can both retry the primary *and* try the fallback chain. |
| **Fallback** | Provider-chain failover | Below Retry so each fallback adapter has its own retry budget. Above RateLimit so each fallback target counts against its own bucket. |
| **RateLimit** | Token-bucket limiter | Innermost — the last gate before the adapter — so every successful call counts the bucket exactly once, regardless of retries from outer layers. |
| **Routing/Adapter** | The actual provider call | Replaces the adapter when classification routing is on; otherwise just the adapter. |

Confirm the canonical order from the live registry. We disable
nothing — every module slot is opted in — so the full stack is
visible. (We mock `_build_adapter` so no network call ever
happens.)

In [ ]:
def stack_layers(provider: LLMProvider) -> list[str]:
    """Walk the _inner chain and return [outer, ..., inner] class names."""
    layers: list[str] = []
    cur: Any = provider
    while cur is not None:
        layers.append(type(cur).__name__)
        cur = getattr(cur, "_inner", None)
    return layers


clear_cache()
with patch("arcllm.registry._build_adapter", return_value=make_mock_inner()):
    full = load_model(
        "anthropic",
        otel={"exporter": "none"},
        queue=True,
        telemetry=True,
        audit=True,
        security=True,
        circuit_breaker=True,
        retry=True,
        fallback={"chain": []},
        rate_limit=True,
    )

for layer in stack_layers(full):
    print(" ", layer)

Read top-to-bottom: that is the **outer-to-inner** order — exactly
the canonical stack. The mock adapter sits at the bottom; every
module wraps the next one down.

A shape sanity check: with no modules enabled, you get the
naked adapter back. With a single module enabled, you get one
wrapper. The stack is always exactly as deep as the number of
enabled modules.

In [ ]:
clear_cache()
with patch("arcllm.registry._build_adapter", return_value=make_mock_inner()):
    bare = load_model(
        "anthropic",
        otel=False,
        queue=False,
        telemetry=False,
        audit=False,
        security=False,
        circuit_breaker=False,
        retry=False,
        fallback=False,
        rate_limit=False,
    )
    only_retry = load_model(
        "anthropic",
        otel=False,
        queue=False,
        telemetry=False,
        audit=False,
        security=False,
        circuit_breaker=False,
        retry=True,
        fallback=False,
        rate_limit=False,
    )
    only_audit = load_model(
        "anthropic",
        otel=False,
        queue=False,
        telemetry=False,
        audit=True,
        security=False,
        circuit_breaker=False,
        retry=False,
        fallback=False,
        rate_limit=False,
    )

print("bare       :", " -> ".join(stack_layers(bare)))
print("retry-only :", " -> ".join(stack_layers(only_retry)))
print("audit-only :", " -> ".join(stack_layers(only_audit)))

## 6. Adding a custom module — `RequestLabelerModule`

Suppose you want every outbound call to carry a label that
identifies which subsystem made it. The label gets injected into
`kwargs` so downstream modules can record it; counters per-label
live on the module instance.

This is a complete, real module. The whole pattern fits in one
cell.

In [ ]:
from arcllm.modules.base import BaseModule, validate_config_keys

_LABELER_KEYS = {"enabled", "label", "counter_attribute"}


class RequestLabelerModule(BaseModule):
    """Stamp every invoke() with a fixed label and count calls.

    Adds two pieces of behaviour around the inner call:
      1. Injects ``request_label`` into kwargs so downstream
         modules can record it.
      2. Increments a per-instance counter so callers can read
         how many calls flowed through.
    """

    def __init__(self, config: dict[str, Any], inner: LLMProvider) -> None:
        super().__init__(config, inner)
        validate_config_keys(config, _LABELER_KEYS, "RequestLabelerModule")
        self._label: str = config.get("label", "unlabeled")
        self._counter_attr: str = config.get("counter_attribute", "call_count")
        if not isinstance(self._label, str) or not self._label:
            raise ArcLLMConfigError("label must be a non-empty string")
        setattr(self, self._counter_attr, 0)

    async def invoke(
        self,
        messages: list[Message],
        tools: list[Tool] | None = None,
        **kwargs: Any,
    ) -> LLMResponse:
        with self._span(
            "arcllm.request_labeler",
            attributes={"arcllm.request_label": self._label},
        ):
            inner_kwargs = {**kwargs, "request_label": self._label}
            response = await self._inner.invoke(messages, tools, **inner_kwargs)
            setattr(self, self._counter_attr, getattr(self, self._counter_attr) + 1)
            return response


print("RequestLabelerModule defined")

Three things to notice:

- The constructor signature is exactly `(self, config, inner)` —
  the same shape every official module uses.
- `validate_config_keys()` rejects typos at construction time.
- The override in `invoke()` adds work *around* the inner call,
  not in place of it. We forward `messages` and `tools` unchanged
  and add a single `request_label` kwarg.

Use it standalone first — wrap a mock inner.

In [ ]:
inner = make_mock_inner()
labeler = RequestLabelerModule({"label": "planner-agent"}, inner)

for _ in range(3):
    await labeler.invoke(messages)

print("call_count:", labeler.call_count)
print("forwarded kwargs (last call):", inner.invoke.await_args.kwargs)

The mock inner records that `request_label="planner-agent"` was
forwarded — exactly the kind of metadata `TelemetryModule` could
now read off and stamp onto a `TraceRecord`.

Reject a bad config to confirm `validate_config_keys()` does its
job:

In [ ]:
try:
    RequestLabelerModule({"lable": "oops"}, make_mock_inner())  # typo!
except ArcLLMConfigError as e:
    print("caught:", e)

try:
    RequestLabelerModule({"label": ""}, make_mock_inner())
except ArcLLMConfigError as e:
    print("caught:", e)

Now slot the labeler into a real stack alongside official modules.
Custom modules don't need registry support — you compose them by
hand. Wrap the result of `load_model(...)` in your custom module
and you're done.

In [ ]:
clear_cache()
with patch("arcllm.registry._build_adapter", return_value=make_mock_inner()):
    inner_stack = load_model(
        "anthropic",
        otel=False,
        queue=False,
        telemetry=False,
        audit=True,
        security=False,
        circuit_breaker=False,
        retry=True,
        fallback=False,
        rate_limit=False,
    )

labeled_stack = RequestLabelerModule(
    {"label": "executor-agent"},
    inner_stack,
)

print("labeled stack:", " -> ".join(stack_layers(labeled_stack)))

for _ in range(2):
    await labeled_stack.invoke(messages)

print("call_count:", labeled_stack.call_count)

The labeler now wraps `RetryModule(AuditModule(adapter))`. Read
the printed stack outer-to-inner: every layer is doing its job
and the labeler is the new outermost. This is the same
composition pattern `load_model()` uses internally — there is
no special registration, no plugin interface, no decorator. Just
subclass `BaseModule` and wrap.

If you want a custom module to behave like a first-class member
of `load_model()`, the path is to (a) subclass `BaseModule`,
(b) add a `[modules.your_module]` block to `arcllm.toml`, and
(c) wrap manually around the result. Most teams stop at (a) +
(c).

## 7. Module ordering matters — `RateLimit` outside `Retry` vs inside

The canonical stack puts `RateLimit` **innermost**. That choice
is observable: move `RateLimit` outside `Retry` and the failure
behaviour changes.

Setup:

- A flaky inner that returns transient `503` errors twice and then
  succeeds — the same kind of pattern `RetryModule` is designed
  for.
- A small token bucket: capacity = 2, refill = 0.

**Canonical (`Retry(RateLimit(adapter))`):** every retry attempt
passes back through the rate limiter on the way to the adapter.
Three attempts cost three tokens.

**Inverted (`RateLimit(Retry(adapter))`):** the rate limiter sees
*one* logical call and consumes one token; retries happen below
it and don't touch the bucket again.

Both stacks return a successful `LLMResponse` — the difference
is how many tokens the bucket lost.

In [ ]:
from arcllm.modules import rate_limit as rl_module


def make_flaky_inner(num_failures: int = 2) -> LLMProvider:
    """Inner that fails the first num_failures calls then succeeds."""
    attempts = {"n": 0}

    async def flaky(*_args, **_kwargs):
        attempts["n"] += 1
        if attempts["n"] <= num_failures:
            raise ArcLLMAPIError(503, "transient", "flaky-provider")
        return LLMResponse(
            content="ok",
            usage=Usage(input_tokens=1, output_tokens=1, total_tokens=2),
            model="flaky-model",
            stop_reason="end_turn",
        )

    inner = MagicMock(spec=LLMProvider)
    inner.name = "flaky-provider"
    inner.model_name = "flaky-model"
    inner.invoke = AsyncMock(side_effect=flaky)
    inner.close = AsyncMock()
    return inner, attempts


RL_CONFIG = {
    "requests_per_minute": 60,
    "burst_capacity": 2,
}
RETRY_CONFIG = {"max_retries": 3, "backoff_base_seconds": 0.001}

print("flaky helper + configs ready")

In [ ]:
# Canonical: Retry(RateLimit(adapter)) — retries attempt to acquire each time.
rl_module.clear_buckets()
inner1, attempts1 = make_flaky_inner(num_failures=2)
rl1 = RateLimitModule(RL_CONFIG, inner1)
canonical = RetryModule(RETRY_CONFIG, rl1)

result = await canonical.invoke(messages)
print("canonical result:", result.content)
print("inner attempts (== retry count):", attempts1["n"])
tokens_left_canonical = rl_module._bucket_registry["flaky-provider"]._tokens
print("tokens left in bucket:", round(tokens_left_canonical, 2))

In [ ]:
# Inverted: RateLimit(Retry(adapter)) — RateLimit only sees the outer call.
rl_module.clear_buckets()
inner2, attempts2 = make_flaky_inner(num_failures=2)
retry2 = RetryModule(RETRY_CONFIG, inner2)
inverted = RateLimitModule(RL_CONFIG, retry2)

result = await inverted.invoke(messages)
print("inverted result:", result.content)
print("inner attempts (== retry count):", attempts2["n"])
tokens_left_inverted = rl_module._bucket_registry["flaky-provider"]._tokens
print("tokens left in bucket:", round(tokens_left_inverted, 2))

Same logical call, same final response, but the bucket
accounting diverges. The canonical order has retries debit the
bucket because each attempt is a real outbound request that
providers count against your rate limit. Inverted order would
*understate* your usage and let you blow through the provider's
true rate cap.

Pick the inverted order on purpose only if you want a soft cap on
logical calls (not physical requests) — and accept that you may
exceed the provider's rate limit during retry storms.

## 8. Stack tracing helpers — walking `_inner` to debug live stacks

`stack_layers()` returns class names; `describe_stack()` is the
fuller version that also dumps each module's recognised config
keys and other quick attributes. Both are < 15 lines of code —
see `arcllm/04-agentic-loop.ipynb` for the equivalent helpers
used there.

When debugging "why does my stack behave like X", the first move
is always: walk `_inner` and print the layers in order. If a
layer you expect is missing, check your kwargs and your
`arcllm.toml`.

In [ ]:
def describe_stack(provider: LLMProvider) -> None:
    """Pretty-print every layer with its config keys and key attrs."""
    cur: Any = provider
    depth = 0
    while cur is not None:
        cls = type(cur).__name__
        cfg = getattr(cur, "_config", None)
        cfg_keys = sorted(cfg.keys()) if isinstance(cfg, dict) else []
        prefix = "  " * depth
        print(f"{prefix}{cls}  config={cfg_keys}")
        cur = getattr(cur, "_inner", None)
        depth += 1


clear_cache()
with patch("arcllm.registry._build_adapter", return_value=make_mock_inner()):
    sample = load_model(
        "anthropic",
        otel={"exporter": "none"},
        queue={"max_concurrent": 2},
        telemetry=True,
        audit=True,
        security=False,
        circuit_breaker={"failure_threshold": 5, "cooldown_seconds": 30.0},
        retry={"max_retries": 2, "backoff_base_seconds": 0.5},
        fallback={"chain": []},
        rate_limit={"requests_per_minute": 60, "burst_capacity": 5},
    )

describe_stack(sample)

The indentation literally walks the `_inner` chain. The
outermost is on the left; each `_inner` is one indent level
deeper. The bottom of the chain (the adapter) has no `_inner`,
so the loop stops.

Wrap the result in your custom module and the helper picks it up
for free — `_inner` is the only convention that matters.

In [ ]:
wrapped = RequestLabelerModule({"label": "diagnostic"}, sample)
describe_stack(wrapped)

## 9. Composition with `load_model()` — quick recap

The full deep-dive on `load_model()`'s kwargs lives in
`arcllm/04-agentic-loop.ipynb`. The short version, since this
notebook has been about the underlying modules:

- Each kwarg toggles one module: `retry`, `fallback`,
  `rate_limit`, `audit`, `security`, `telemetry`, `otel`,
  `queue`, `circuit_breaker`, `routing`.
- Each kwarg accepts `True` (use `arcllm.toml` defaults), `False`
  (force off), a `dict` (force on, merged over defaults), or
  `None` (use `enabled` flag in TOML).
- Three threading kwargs feed callbacks into the stack:
  `on_event`, `trace_store`, `agent_label`. Plus `budget_scope`
  for telemetry budget tracking.
- The wrap order is fixed and matches the canonical stack.

Everything you saw in this notebook (`_inner` walking, custom
modules, ordering effects) is true regardless of how the stack
was built. `load_model()` is just the convenient front door.

In [ ]:
clear_cache()
with patch("arcllm.registry._build_adapter", return_value=make_mock_inner()):
    minimal = load_model(
        "anthropic",
        retry=True,
        audit=True,
        otel=False,
        queue=False,
        telemetry=False,
        security=False,
        circuit_breaker=False,
        fallback=False,
        rate_limit=False,
    )
    full = load_model(
        "anthropic",
        otel={"exporter": "none"},
        queue=True,
        telemetry=True,
        audit=True,
        security=True,
        circuit_breaker=True,
        retry=True,
        fallback={"chain": []},
        rate_limit=True,
    )

print("minimal:", " -> ".join(stack_layers(minimal)))
print("full   :", " -> ".join(stack_layers(full)))

Same `load_model()` call, two stacks — kwargs decide what's
wrapped. The order *within* each stack is identical to the
canonical order, just with the disabled layers omitted.

One more reusability win: every module is a working `LLMProvider`,
so you can hand a stack-fragment back to anything that takes an
`LLMProvider`. For example, `FallbackModule` accepts adapter
names but it could just as well accept fully-wrapped sub-stacks
if you wanted per-fallback retry policies.

In [ ]:
# Demonstrate: invoke against the full stack — every layer participates,
# the response is unchanged, and the inner mock recorded one call.
result = await full.invoke(messages)
print("result content:", result.content)

## 10. Summary

**`BaseModule` is the contract.** Every module in arcllm — official
or custom — subclasses it. The contract is small on purpose:
store `_inner`, override `invoke()`, optionally override
`close()`, and call `validate_config_keys()` in `__init__` to
catch typos. There is no lifecycle, no plugin registry, no
decorators — just a subclass and `super().__init__(config, inner)`.

**Eleven modules ship with v0.4.0.** `OtelModule`,
`QueueModule`, `TelemetryModule`, `AuditModule`,
`SecurityModule`, `CircuitBreakerModule`, `RetryModule`,
`FallbackModule`, `RateLimitModule`, `RoutingModule`, and
`BaseModule` itself. Each one has a dedicated notebook (08–12,
15, 17) for its config keys, OTel attributes, and edge cases.

**The canonical stack order is fixed and intentional.**
`Otel → Queue → Telemetry → Audit → Security → CircuitBreaker →
Retry → Fallback → RateLimit → [Routing|Adapter]`. Each layer's
position reflects what it needs to observe and what it needs to
protect. Section 7 showed the consequence: invert `RateLimit` and
`Retry` and your bucket accounting silently understates real
provider usage.

**Custom modules are subclasses, not plugins.** Section 6 wrote
`RequestLabelerModule` in fifteen lines and slotted it into a
live stack with no registry support. The same pattern is how
every official module is written — they just live in
`arcllm/modules/`.

**Walking `_inner` is the universal debugging move.**
`stack_layers()` and `describe_stack()` are < 15 lines each and
work on any provider that follows the convention. When in doubt,
print the stack.

Where to go next:

- `arcllm/04-agentic-loop.ipynb` — every kwarg of `load_model()`
  in detail, including `on_event`, `trace_store`, `agent_label`.
- `arcllm/08-rate-limiter.ipynb` — `RateLimitModule` deep dive.
- `arcllm/09-telemetry-module.ipynb` — `TelemetryModule` and
  phase timings.
- `arcllm/10-audit-trail.ipynb` — `AuditModule` and arctrust
  audit sinks.
- `arcllm/11-otel-export.ipynb` — `OtelModule` span attributes
  and exporters.
- `arcllm/12-security-layer.ipynb` — `SecurityModule` PII
  redaction and request signing.
- `arcllm/15-queue-circuit-breaker.ipynb` — `QueueModule` and
  `CircuitBreakerModule` end-to-end with concurrency tests.
- `arcllm/17-routing-module.ipynb` — `RoutingModule`
  classification taxonomy and enforcement modes.

Public API exercised in this notebook:

`arcllm.BaseModule`, `arcllm.AuditModule`, `arcllm.FallbackModule`,
`arcllm.OtelModule`, `arcllm.QueueModule`, `arcllm.RateLimitModule`,
`arcllm.RetryModule`, `arcllm.SecurityModule`,
`arcllm.TelemetryModule`,
`arcllm.modules.circuit_breaker.CircuitBreakerModule`,
`arcllm.modules.routing.RoutingModule`,
`arcllm.modules.base.validate_config_keys`,
`arcllm.load_model`, `arcllm.clear_cache`,
`arcllm.types.LLMProvider`, plus the data types `Message`,
`Tool`, `LLMResponse`, `Usage`, and the exceptions
`ArcLLMAPIError`, `ArcLLMConfigError`.